In [13]:
library(data.table)

# =========================================================
# STEP 1: Load base dataset with anomalies + meteo lags
# Filter selected IGBP classes
# Identify sites with continuous 3-year records
# =========================================================

base_file <- "derived_tables/modelready_with_anomalies_gscenter_meteolags_2016_2024_clean.csv"

dt_base <- fread(base_file)

cat("Loaded base dataset:\n")
cat("Rows :", nrow(dt_base), "\n")
cat("Cols :", ncol(dt_base), "\n")

# ---------------------------------------------------------
# Basic column checks
# ---------------------------------------------------------

cat("\nColumn names containing site/year info:\n")
print(grep("site|SITE|year|YEAR|IGBP", names(dt_base), value = TRUE))

# Standardize SITE_ID if needed
if ("site_id" %in% names(dt_base) && !("SITE_ID" %in% names(dt_base))) {
  setnames(dt_base, "site_id", "SITE_ID")
}

if ("Site_ID" %in% names(dt_base) && !("SITE_ID" %in% names(dt_base))) {
  setnames(dt_base, "Site_ID", "SITE_ID")
}

if ("YEAR" %in% names(dt_base) && !("year" %in% names(dt_base))) {
  setnames(dt_base, "YEAR", "year")
}

if ("Year" %in% names(dt_base) && !("year" %in% names(dt_base))) {
  setnames(dt_base, "Year", "year")
}

stopifnot("SITE_ID" %in% names(dt_base))
stopifnot("year" %in% names(dt_base))
stopifnot("IGBP" %in% names(dt_base))

dt_base[, year := as.integer(year)]
setorder(dt_base, SITE_ID, year)

cat("\nAfter standardizing names:\n")
cat("Sites:", uniqueN(dt_base$SITE_ID), "\n")
cat("Years:", min(dt_base$year, na.rm = TRUE), "-", max(dt_base$year, na.rm = TRUE), "\n")

# ---------------------------------------------------------
# Keep only selected IGBP classes
# ---------------------------------------------------------

igbp_keep <- c(
  "CSH", "DBF", "DNF", "EBF", "ENF",
  "MF", "OSH", "SAV", "WET", "WSA"
)

cat("\nAvailable IGBP classes before filter:\n")
print(dt_base[, .N, by = IGBP][order(IGBP)])

cat("\nBefore IGBP filter:\n")
cat("Rows :", nrow(dt_base), "\n")
cat("Sites:", uniqueN(dt_base$SITE_ID), "\n")

dt_base <- dt_base[IGBP %in% igbp_keep]

cat("\nAfter IGBP filter:\n")
cat("Rows :", nrow(dt_base), "\n")
cat("Sites:", uniqueN(dt_base$SITE_ID), "\n")

cat("\nRemaining IGBP classes:\n")
print(dt_base[, .N, by = IGBP][order(IGBP)])

# ---------------------------------------------------------
# Check continuous 3-year availability
# For each row/year t, require t-1 and t-2 for same site
# ---------------------------------------------------------

site_years <- unique(dt_base[, .(SITE_ID, year)])

dt_base[, has_tminus1 := paste(SITE_ID, year - 1) %in% paste(site_years$SITE_ID, site_years$year)]
dt_base[, has_tminus2 := paste(SITE_ID, year - 2) %in% paste(site_years$SITE_ID, site_years$year)]

dt_3yr <- dt_base[has_tminus1 == TRUE & has_tminus2 == TRUE]

cat("\nContinuous 3-year filter result:\n")
cat("Rows before:", nrow(dt_base), "\n")
cat("Sites before:", uniqueN(dt_base$SITE_ID), "\n")
cat("Rows after :", nrow(dt_3yr), "\n")
cat("Sites after:", uniqueN(dt_3yr$SITE_ID), "\n")

cat("\nRows per year after filter:\n")
print(dt_3yr[, .N, by = year][order(year)])

cat("\nRows per site after filter, first 20:\n")
print(dt_3yr[, .N, by = SITE_ID][order(-N)][1:20])

# ---------------------------------------------------------
# Check response variables
# ---------------------------------------------------------

response_vars <- c("uWUE", "ETmax", "GPPsat", "NEPmax")

cat("\nResponse availability after 3-year filter:\n")
for (v in response_vars) {
  if (v %in% names(dt_3yr)) {
    cat(v, ": non-NA =", sum(!is.na(dt_3yr[[v]])),
        "| NA =", sum(is.na(dt_3yr[[v]])), "\n")
  } else {
    cat(v, ": NOT FOUND\n")
  }
}

Loaded base dataset:
Rows : 701 
Cols : 497 

Column names containing site/year info:
[1] "SITE_ID"          "year"             "nyears"           "IGBP"            
[5] "uWUE_site_mean"   "ETmax_site_mean"  "GPPsat_site_mean" "NEPmax_site_mean"

After standardizing names:
Sites: 142 
Years: 2016 - 2024 

Available IGBP classes before filter:
      IGBP     N
    <char> <int>
 1:    CSH    24
 2:    CVM    21
 3:    DBF   135
 4:    DNF     8
 5:    EBF    26
 6:    ENF   180
 7:     MF    24
 8:    OSH    87
 9:    SAV    36
10:    WET   138
11:    WSA    22

Before IGBP filter:
Rows : 701 
Sites: 142 

After IGBP filter:
Rows : 680 
Sites: 138 

Remaining IGBP classes:
      IGBP     N
    <char> <int>
 1:    CSH    24
 2:    DBF   135
 3:    DNF     8
 4:    EBF    26
 5:    ENF   180
 6:     MF    24
 7:    OSH    87
 8:    SAV    36
 9:    WET   138
10:    WSA    22

Continuous 3-year filter result:
Rows before: 680 
Sites before: 138 
Rows after : 410 
Sites after: 103 

Rows per

In [14]:
library(reticulate)

use_condaenv("clean_r_env", required = TRUE)

py_config()

python:         /home/nk1125/miniconda3/envs/clean_r_env/bin/python
libpython:      /home/nk1125/miniconda3/envs/clean_r_env/lib/libpython3.13.so
pythonhome:     /home/nk1125/miniconda3/envs/clean_r_env:/home/nk1125/miniconda3/envs/clean_r_env
version:        3.13.3 | packaged by conda-forge | (main, Apr 14 2025, 20:44:03) [GCC 13.3.0]
numpy:          /home/nk1125/miniconda3/envs/clean_r_env/lib/python3.13/site-packages/numpy
numpy_version:  2.4.4

NOTE: Python version was forced by use_python() function

In [15]:

py_install(c("xarray", "zarr", "numpy"), pip = TRUE)

In [16]:
py_require(c("xarray", "zarr", "numpy"))

In [17]:
xr <- import("xarray", delay_load = FALSE)

zarr_dir <- file.path("deadtree", "sentinelcubes")
zarr_paths <- list.files(zarr_dir, pattern = "_inference\\.zarr$", full.names = TRUE)


In [18]:
library(data.table)
library(dplyr)
library(stringr)

# =========================================================
# STEP 2: Recalculate disturbance for 103 valid sites
# Based on dt_3yr from STEP 1
# =========================================================

out_dir <- "derived_tables/final_results_3yr_clean_103sites"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# ---------------------------------------------------------
# Get final valid sites from STEP 1 result
# ---------------------------------------------------------

final_sites <- sort(unique(dt_3yr$SITE_ID))

cat("Final sites from filtered 3-year dataset:", length(final_sites), "\n")

writeLines(
  final_sites,
  file.path(out_dir, "valid_103_sites_3yr_IGBP_filtered.txt")
)

# ---------------------------------------------------------
# Match zarr paths to valid sites
# ---------------------------------------------------------

get_site_id <- function(zpath) {
  str_replace(basename(zpath), "_inference\\.zarr$", "")
}

zarr_site_ids <- vapply(zarr_paths, get_site_id, character(1))

zarr_paths_valid <- zarr_paths[zarr_site_ids %in% final_sites]

cat("Zarr files matched:", length(zarr_paths_valid), "\n")
cat("Missing zarr files:", length(setdiff(final_sites, zarr_site_ids)), "\n")

missing_zarr_sites <- setdiff(final_sites, zarr_site_ids)

writeLines(
  missing_zarr_sites,
  file.path(out_dir, "missing_zarr_sites.txt")
)

# ---------------------------------------------------------
# Buffer radii
# ---------------------------------------------------------

buffer_radii <- c(100, 200, 300, 400, 500)

# ---------------------------------------------------------
# Resume checkpoint if available
# ---------------------------------------------------------

checkpoint_rdata <- file.path(out_dir, "checkpoint_workspace_103sites.RData")
checkpoint_csv   <- file.path(out_dir, "checkpoint_disturbance_103sites_multibuffer.csv")
failed_file      <- file.path(out_dir, "failed_sites_103sites.txt")

if (file.exists(checkpoint_rdata)) {
  load(checkpoint_rdata)
  message("Loaded existing checkpoint workspace.")
} else {
  all_results <- list()
  failed_sites <- character()
}

done_sites <- names(all_results)

zarr_paths_run <- zarr_paths_valid[
  !(vapply(zarr_paths_valid, get_site_id, character(1)) %in% done_sites)
]

message("Sites already done: ", length(done_sites))
message("Sites left to run: ", length(zarr_paths_run))

# =========================================================
# Loop over sites
# =========================================================

for (zpath in zarr_paths_run) {
  
  site_id <- get_site_id(zpath)
  message("Processing ", site_id)
  
  res <- tryCatch({
    
    ds <- xr$open_zarr(zpath, consolidated = FALSE)
    
    years <- as.integer(py_to_r(ds[["time"]]$values))
    x <- as.vector(py_to_r(ds[["x"]]$values))
    y <- as.vector(py_to_r(ds[["y"]]$values))
    
    # ------------------------------------------------------
    # Build distance grid from cube center
    # ------------------------------------------------------
    
    x0 <- mean(range(x))
    y0 <- mean(range(y))
    dist2 <- outer((y - y0)^2, (x - x0)^2, "+")
    
    mask_list <- lapply(buffer_radii, function(r) dist2 <= (r^2))
    names(mask_list) <- paste0(buffer_radii, "m")
    
    # ------------------------------------------------------
    # Load forest and deadwood arrays and convert to %
    # expected dimensions: [time, y, x]
    # ------------------------------------------------------
    
    forest_arr <- py_to_r(ds[["forest"]]$values)
    forest_arr <- array(as.numeric(forest_arr) * (100 / 255), dim = dim(forest_arr))
    
    deadwood_arr <- py_to_r(ds[["deadwood"]]$values)
    deadwood_arr <- array(as.numeric(deadwood_arr) * (100 / 255), dim = dim(deadwood_arr))
    
    out_list <- vector("list", length(years))
    
    for (i in seq_along(years)) {
      
      forest_now <- forest_arr[i, , ]
      deadwood_now <- deadwood_arr[i, , ]
      
      row_out <- list(
        SITE_ID = site_id,
        year = years[i]
      )
      
      if (i > 1) {
        forest_prev <- forest_arr[i - 1, , ]
        deadwood_prev <- deadwood_arr[i - 1, , ]
        
        forest_delta <- forest_now - forest_prev
        deadwood_delta <- deadwood_now - deadwood_prev
        
        loss_pp <- ifelse(forest_delta <= -20, -forest_delta, 0)
        deadwood_gain_pp <- ifelse(deadwood_delta >= 20, deadwood_delta, 0)
      }
      
      for (r in buffer_radii) {
        
        rlab <- paste0(r, "m")
        mask <- mask_list[[rlab]]
        
        # forest_mean_pct kept for possible context/sensitivity,
        # but you can exclude it later from the main disturbance model.
        row_out[[paste0("forest_mean_pct_", rlab)]] <- mean(forest_now[mask], na.rm = TRUE)
        row_out[[paste0("deadwood_mean_pct_", rlab)]] <- mean(deadwood_now[mask], na.rm = TRUE)
        
        if (i == 1) {
          
          row_out[[paste0("loss_area_frac_", rlab)]] <- NA_real_
          row_out[[paste0("loss_mean_pp_", rlab)]]   <- NA_real_
          row_out[[paste0("loss_sum_pp_", rlab)]]    <- NA_real_
          
          row_out[[paste0("deadwood_increase_area_frac_", rlab)]] <- NA_real_
          row_out[[paste0("deadwood_increase_mean_pp_", rlab)]]   <- NA_real_
          row_out[[paste0("deadwood_increase_sum_pp_", rlab)]]    <- NA_real_
          
        } else {
          
          masked_loss <- loss_pp[mask]
          masked_deadwood_gain <- deadwood_gain_pp[mask]
          
          row_out[[paste0("loss_area_frac_", rlab)]] <- mean(masked_loss > 0, na.rm = TRUE)
          row_out[[paste0("loss_mean_pp_", rlab)]]   <- mean(masked_loss, na.rm = TRUE)
          row_out[[paste0("loss_sum_pp_", rlab)]]    <- sum(masked_loss, na.rm = TRUE)
          
          row_out[[paste0("deadwood_increase_area_frac_", rlab)]] <- mean(masked_deadwood_gain > 0, na.rm = TRUE)
          row_out[[paste0("deadwood_increase_mean_pp_", rlab)]]   <- mean(masked_deadwood_gain, na.rm = TRUE)
          row_out[[paste0("deadwood_increase_sum_pp_", rlab)]]    <- sum(masked_deadwood_gain, na.rm = TRUE)
        }
      }
      
      out_list[[i]] <- as.data.frame(row_out, check.names = FALSE)
    }
    
    bind_rows(out_list)
    
  }, error = function(e) {
    message("Failed for ", site_id, ": ", conditionMessage(e))
    failed_sites <<- unique(c(failed_sites, site_id))
    return(NULL)
  })
  
  if (!is.null(res) && nrow(res) > 0) {
    
    all_results[[site_id]] <- res
    
    partial_df <- bind_rows(all_results)
    
    fwrite(partial_df, checkpoint_csv)
    writeLines(failed_sites, failed_file)
    save.image(checkpoint_rdata)
    
    message("Checkpoint saved after site ", site_id)
  }
}

# =========================================================
# Final output
# =========================================================

final_disturbance_df <- bind_rows(all_results)

fwrite(
  final_disturbance_df,
  file.path(out_dir, "final_disturbance_103sites_multibuffer.csv")
)

writeLines(failed_sites, failed_file)

save.image(
  file.path(out_dir, "final_workspace_103sites_multibuffer.RData")
)

message("Done.")
message("Final rows: ", nrow(final_disturbance_df))
message("Final sites: ", uniqueN(final_disturbance_df$SITE_ID))
message("Failed sites: ", length(failed_sites))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Final sites from filtered 3-year dataset: 103 
Zarr files matched: 103 
Missing zarr files: 0 


Sites already done: 0

Sites left to run: 103

Processing AU-Boy

Checkpoint saved after site AU-Boy

Processing AU-GWW

Checkpoint saved after site AU-War

Processing AU-Whr

Checkpoint saved after site AU-Whr

Processing AU-Wom

Checkpoint saved after site AU-Wom

Processing CA-Cbo

Checkpoint saved after site CA-Cbo

Processing CA-EM1

Checkpoint saved after site CA-LP1

Processing CA-TPD

Checkpoint saved after site CA-TPD

Processing CL-SDF

Checkpoint saved after site CL-SDF

Processing CZ-RAJ

Checkpoint saved after site CZ-RAJ

Processing CZ-Stn

Checkpoint saved after site CZ-Stn

Processing DE-Akm

Checkpoint saved after site DE-Akm

Processing DE-Har

Checkpoint saved after site DE-Har

Processing DE-Hte

Checkpoint saved after site DE-Hte

Processing DE-Msr

Checkpoint saved after site DK-Sor

Processing ES-Abr

Checkpoint saved after site ES-Abr

Processing ES-Agu

Checkpoint saved after site ES-Agu

Processing ES-LJu

Checkpoint saved after site ES-LJu

Processing ES-LM1


In [20]:
# =========================================================
# REMOVE old disturbance columns before merge
# VERY IMPORTANT
# =========================================================

old_disturbance_patterns <- c(
  "^forest_mean_pct_[0-9]+m$",
  "^loss_area_frac_[0-9]+m$",
  "^loss_mean_pp_[0-9]+m$",
  "^loss_sum_pp_[0-9]+m$",
  "^deadwood_mean_pct_[0-9]+m$",
  "^deadwood_increase_area_frac_[0-9]+m$",
  "^deadwood_increase_mean_pp_[0-9]+m$",
  "^deadwood_increase_sum_pp_[0-9]+m$"
)

old_disturbance_cols <- unique(unlist(
  lapply(old_disturbance_patterns, function(p) {
    grep(p, names(dt_3yr), value = TRUE)
  })
))

cat("Old disturbance columns already in dt_3yr:",
    length(old_disturbance_cols), "\n")

print(old_disturbance_cols)

if (length(old_disturbance_cols) > 0) {
  dt_3yr[, (old_disturbance_cols) := NULL]
}

cat("Columns after removal:", ncol(dt_3yr), "\n")

Old disturbance columns already in dt_3yr: 4 
[1] "loss_area_frac_500m"    "loss_mean_pp_500m"      "loss_sum_pp_500m"      
[4] "deadwood_mean_pct_500m"
Columns after removal: 495 


In [ ]:
library(data.table)

# =========================================================
# STEP 3: Merge recalculated disturbance with filtered base data
# and create disturbance t, t-1, t-2
# =========================================================

out_dir <- "derived_tables/final_results_3yr_clean_103sites"
dist_file <- file.path(out_dir, "final_disturbance_103sites_multibuffer.csv")

dist_dt <- fread(dist_file)

cat("Loaded disturbance:\n")
cat("Rows :", nrow(dist_dt), "\n")
cat("Sites:", uniqueN(dist_dt$SITE_ID), "\n")

# ---------------------------------------------------------
# Detect disturbance and forest-context variables separately
# ---------------------------------------------------------

forest_context_patterns <- c("^forest_mean_pct_[0-9]+m$")

disturbance_patterns <- c(
  "^loss_area_frac_[0-9]+m$",
  "^loss_mean_pp_[0-9]+m$",
  "^loss_sum_pp_[0-9]+m$",
  "^deadwood_mean_pct_[0-9]+m$",
  "^deadwood_increase_area_frac_[0-9]+m$",
  "^deadwood_increase_mean_pp_[0-9]+m$",
  "^deadwood_increase_sum_pp_[0-9]+m$"
)

forest_context_vars <- unique(unlist(lapply(forest_context_patterns, function(p) {
  grep(p, names(dist_dt), value = TRUE)
})))

disturbance_vars <- unique(unlist(lapply(disturbance_patterns, function(p) {
  grep(p, names(dist_dt), value = TRUE)
})))

cat("\nForest context vars:", length(forest_context_vars), "\n")
cat("Disturbance vars   :", length(disturbance_vars), "\n")

# ---------------------------------------------------------
# Keep only needed disturbance columns
# ---------------------------------------------------------

dist_keep <- c("SITE_ID", "year", forest_context_vars, disturbance_vars)
dist_dt <- dist_dt[, ..dist_keep]

dist_dt[, year := as.integer(year)]
dt_3yr[, year := as.integer(year)]

# ---------------------------------------------------------
# Remove old disturbance/forest columns from dt_3yr before merge
# ---------------------------------------------------------

old_patterns <- c(forest_context_patterns, disturbance_patterns)

old_cols <- unique(unlist(lapply(old_patterns, function(p) {
  grep(p, names(dt_3yr), value = TRUE)
})))

cat("\nOld disturbance/context cols already in dt_3yr:", length(old_cols), "\n")

if (length(old_cols) > 0) {
  dt_3yr[, (old_cols) := NULL]
}

cat("dt_3yr columns after removing old disturbance/context:", ncol(dt_3yr), "\n")

# ---------------------------------------------------------
# Merge current-year disturbance into filtered base data
# ---------------------------------------------------------

dt_model <- merge(
  dt_3yr,
  dist_dt,
  by = c("SITE_ID", "year"),
  all.x = TRUE
)

setorder(dt_model, SITE_ID, year)

cat("\nAfter merging current disturbance:\n")
cat("Rows :", nrow(dt_model), "\n")
cat("Sites:", uniqueN(dt_model$SITE_ID), "\n")
cat("Cols :", ncol(dt_model), "\n")

# safety check: no .x/.y duplicate columns
dup_suffix_cols <- grep("\\.x$|\\.y$", names(dt_model), value = TRUE)
cat("Columns with .x/.y suffix:", length(dup_suffix_cols), "\n")
if (length(dup_suffix_cols) > 0) print(dup_suffix_cols)

# ---------------------------------------------------------
# Create exact-year disturbance lags: t-1 and t-2
# ---------------------------------------------------------

disturbance_lag_vars <- character()

for (v in disturbance_vars) {  
  if (!(v %in% names(dt_model))) {
    stop("Missing after merge: ", v)
  }  
  base <- dt_model[, .(
    SITE_ID,
    year,
    value = get(v)
  )]
  for (lag_year in 1:2) {    
    lag_name <- paste0(v, "_lag", lag_year, "yr")    
    tmp <- copy(base)
    tmp[, year := year + lag_year]
    setnames(tmp, "value", lag_name)    
    dt_model <- merge(
      dt_model,
      tmp,
      by = c("SITE_ID", "year"),
      all.x = TRUE
    )   
    disturbance_lag_vars <- c(disturbance_lag_vars, lag_name)
  }
}

setorder(dt_model, SITE_ID, year)

cat("\nCreated disturbance lag vars:", length(disturbance_lag_vars), "\n")

# ---------------------------------------------------------
# NA diagnostics
# ---------------------------------------------------------

disturbance_all <- c(disturbance_vars, disturbance_lag_vars)

na_check <- data.table(
  variable = disturbance_all,
  n_NA = sapply(disturbance_all, function(v) sum(is.na(dt_model[[v]]))),
  pct_NA = sapply(disturbance_all, function(v) 100 * mean(is.na(dt_model[[v]])))
)

setorder(na_check, -n_NA)

cat("\nNA check for disturbance t, t-1, t-2:\n")
print(na_check)

cat("\nRows with complete disturbance t/t-1/t-2:",
    sum(complete.cases(dt_model[, ..disturbance_all])), "\n")

cat("Sites with complete disturbance t/t-1/t-2:",
    uniqueN(dt_model[complete.cases(dt_model[, ..disturbance_all]), SITE_ID]), "\n")

# ---------------------------------------------------------
# Save merged model dataset
# ---------------------------------------------------------

fwrite(
  dt_model,
  file.path(out_dir, "modeldata_103sites_IGBP_3yr_anom_meteolags_disturbance_t_tminus1_tminus2.csv")
)

fwrite(
  na_check,
  file.path(out_dir, "NA_check_disturbance_t_tminus1_tminus2.csv")
)

writeLines(disturbance_vars, file.path(out_dir, "vars_disturbance_current_used.txt"))
writeLines(disturbance_lag_vars, file.path(out_dir, "vars_disturbance_lag1_lag2_used.txt"))
writeLines(forest_context_vars, file.path(out_dir, "vars_forest_context_used.txt"))

cat("\nSaved model dataset to:\n")
cat(file.path(out_dir, "modeldata_103sites_IGBP_3yr_anom_meteolags_disturbance_t_tminus1_tminus2.csv"), "\n")

Loaded disturbance:
Rows : 956 
Sites: 103 

Forest context vars: 5 
Disturbance vars   : 35 

Old disturbance/context cols already in dt_3yr: 0 
dt_3yr columns after removing old disturbance/context: 495 

After merging current disturbance:
Rows : 410 
Sites: 103 
Cols : 535 
Columns with .x/.y suffix: 1 
[1] "forest_mean_pct_500m.x"

Created disturbance lag vars: 70 

NA check for disturbance t, t-1, t-2:
                          variable  n_NA   pct_NA
                            <char> <int>    <num>
  1:    loss_area_frac_100m_lag2yr   201 49.02439
  2:    loss_area_frac_200m_lag2yr   201 49.02439
  3:    loss_area_frac_300m_lag2yr   201 49.02439
  4:    loss_area_frac_400m_lag2yr   201 49.02439
  5:    loss_area_frac_500m_lag2yr   201 49.02439
 ---                                             
101: deadwood_increase_sum_pp_100m     0  0.00000
102: deadwood_increase_sum_pp_200m     0  0.00000
103: deadwood_increase_sum_pp_300m     0  0.00000
104: deadwood_increase_sum_pp_400m     

In [22]:
# =========================================================
# STEP 3b: Clean final model dataset
# remove duplicate .x/.y columns and keep complete disturbance t/t-1/t-2
# =========================================================

out_dir <- "derived_tables/final_results_3yr_clean_103sites"

dt_model <- fread(
  file.path(out_dir, "modeldata_103sites_IGBP_3yr_anom_meteolags_disturbance_t_tminus1_tminus2.csv")
)

disturbance_vars <- readLines(file.path(out_dir, "vars_disturbance_current_used.txt"))
disturbance_lag_vars <- readLines(file.path(out_dir, "vars_disturbance_lag1_lag2_used.txt"))

disturbance_all <- c(disturbance_vars, disturbance_lag_vars)

# remove any leftover duplicate merge columns
dup_suffix_cols <- grep("\\.x$|\\.y$", names(dt_model), value = TRUE)

cat("Removing duplicate suffix columns:", length(dup_suffix_cols), "\n")
print(dup_suffix_cols)

if (length(dup_suffix_cols) > 0) {
  dt_model[, (dup_suffix_cols) := NULL]
}

# keep rows complete for disturbance t, t-1, t-2
dt_model_complete <- dt_model[
  complete.cases(dt_model[, ..disturbance_all])
]

cat("\nFinal complete disturbance dataset:\n")
cat("Rows :", nrow(dt_model_complete), "\n")
cat("Sites:", uniqueN(dt_model_complete$SITE_ID), "\n")
cat("Cols :", ncol(dt_model_complete), "\n")

cat("\nRows per year:\n")
print(dt_model_complete[, .N, by = year][order(year)])

cat("\nRows per IGBP:\n")
print(dt_model_complete[, .N, by = IGBP][order(IGBP)])

# save clean dataset
fwrite(
  dt_model_complete,
  file.path(out_dir, "modeldata_83sites_complete_disturbance_t_tminus1_tminus2.csv")
)

cat("\nSaved clean final dataset:\n")
cat(file.path(out_dir, "modeldata_83sites_complete_disturbance_t_tminus1_tminus2.csv"), "\n")

Removing duplicate suffix columns: 1 
[1] "forest_mean_pct_500m.x"

Final complete disturbance dataset:
Rows : 209 
Sites: 83 
Cols : 604 

Rows per year:
    year     N
   <int> <int>
1:  2020    51
2:  2021    37
3:  2022    34
4:  2023    48
5:  2024    39

Rows per IGBP:
     IGBP     N
   <char> <int>
1:    CSH    12
2:    DBF    39
3:    EBF     6
4:    ENF    59
5:     MF     8
6:    OSH    28
7:    SAV    12
8:    WET    36
9:    WSA     9

Saved clean final dataset:
derived_tables/final_results_3yr_clean_103sites/modeldata_83sites_complete_disturbance_t_tminus1_tminus2.csv 
